# Maternal Health Risk — Analysis and Interactive Dashboard

This notebook reproduces the original analysis and provides an interactive dashboard (Dash) plus a standalone HTML fallback. Run cells in order. If you cannot run the Dash server, use the exported `maternal_dashboard.html` file produced by the fallback cell.

In [ ]:
# Install dependencies (uncomment and run if needed)
# import sys
# !
 -m pip install pandas scikit-learn imbalanced-learn plotly dash==2.9.3

In [ ]:
# Standard imports
import os
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

In [ ]:
# Load dataset (adjust path if needed).
DATA_PATH = os.path.join('public','projects','maternal_health_risk_predictor','Maternal Health Risk Data Set.csv')
df = pd.read_csv(DATA_PATH)
print('Rows,cols:', df.shape)
df.head()

In [ ]:
# Quick EDA
print('Nulls by column:
', df.isnull().sum())
print('
Value counts for RiskLevel:')
print(df['RiskLevel'].value_counts())
# Basic numeric summary
df.describe().T

In [ ]:
# Prepare features and target
y = df['RiskLevel']
X = df.drop(columns=['RiskLevel'])
# Encode target to integers
le = LabelEncoder()
y_enc = le.fit_transform(y)
print('Classes:', list(le.classes_))
# Normalize numeric columns if present
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
scaler = MinMaxScaler()
if len(num_cols):
    X[num_cols] = scaler.fit_transform(X[num_cols])
X.head()

In [ ]:
# Handle imbalance with SMOTE and train a baseline Logistic Regression
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y_enc)
X_train, X_test, y_train, y_test = train_test_split(X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print('Logistic Regression accuracy (after SMOTE):', acc)
print('
Classification report:
', classification_report(y_test, y_pred))
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
# Feature importance (approx) using coefficients for linear model
if hasattr(model, 'coef_'):
    coefs = model.coef_
    # For multiclass, take mean abs coef per feature
    imp = np.mean(np.abs(coefs), axis=0)
    feat_imp = pd.DataFrame({'feature': X.columns, 'importance': imp}).sort_values('importance', ascending=False)
    fig_imp = px.bar(feat_imp, x='feature', y='importance', title='Feature importance (mean |coef|)')
    fig_imp.show()
else:
    print('Model has no coef_ to estimate importances')

In [ ]:
# Create interactive Dash app (run this cell to start the server).
# NOTE: Running the cell will start a Dash server that blocks the notebook kernel until stopped.
try:
    from dash import Dash, dcc, html, Input, Output
    import dash
except Exception as e:
    print('Dash not installed:', e)

def create_dash(app_port=8050):
    app = Dash(__name__)
    # Prepare options
    model_options = ['LogisticRegression']
    class_names = list(le.classes_)
    app.layout = html.Div([
        html.H2('Maternal Health Risk Dashboard'),
        html.Div([
            html.Label('Select metric / view'),
            dcc.Dropdown(id='view-dropdown', options=[
                {'label': 'Class distribution', 'value': 'dist'},
                {'label': 'Feature importance', 'value': 'imp'},
                {'label': 'Confusion matrix', 'value': 'cm'}
            ], value='dist')
        ], style={'width':'300px'}),
        dcc.Graph(id='main-graph'),
        html.Div(id='metrics-output')
    ])

    @app.callback(Output('main-graph', 'figure'), Output('metrics-output','children'), Input('view-dropdown','value'))
    def update_view(view):
        if view == 'dist':
            fig = px.bar(df['RiskLevel'].value_counts().reset_index().rename(columns={'index':'RiskLevel','RiskLevel':'count'}), x='RiskLevel', y='count', title='Class distribution')
            return fig, ''
        elif view == 'imp':
            if 'feat_imp' in globals():
                fig = px.bar(feat_imp, x='feature', y='importance', title='Feature importance')
                return fig, ''
            return go.Figure(), 'No feature importance available'
        elif view == 'cm':
            cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
            fig = px.imshow(cm_df, text_auto=True, title='Confusion Matrix')
            metrics = f'Accuracy: {acc:.4f}'
            return fig, metrics
        return go.Figure(), ''

    return app

# To run the dashboard, uncomment and execute the following (it will block the kernel):
# app = create_dash()
# app.run_server(debug=True, port=8050)

print('Dash app factory created. To start: run create_dash() and app.run_server() in a code cell.')

In [ ]:
# HTML fallback: generate standalone Plotly HTML with key charts (class distribution + feature importance + confusion matrix)
from plotly.subplots import make_subplots
from plotly.offline import plot
figs = []
fig1 = px.bar(df['RiskLevel'].value_counts().reset_index().rename(columns={'index':'RiskLevel','RiskLevel':'count'}), x='RiskLevel', y='count', title='Class distribution')
figs.append(fig1)
if 'feat_imp' in globals():
    figs.append(px.bar(feat_imp, x='feature', y='importance', title='Feature importance'))
# Confusion matrix heatmap as figure
cm_df = pd.DataFrame(cm, index=list(le.classes_), columns=list(le.classes_))
figs.append(px.imshow(cm_df, text_auto=True, title='Confusion matrix'))
outpath = os.path.join('public','projects','maternal_health_risk_predictor','maternal_dashboard.html')
# Combine into a single HTML file by writing each figure sequentially
html_parts = []
for f in figs:
    html_parts.append(plot(f, include_plotlyjs='cdn', output_type='div'))
full_html = '<html><head><meta charset=
><title>Maternal Dashboard</title></head><body>' + '
'.join(html_parts) + '</body></html>'
with open(outpath, 'w', encoding='utf-8') as fh:
    fh.write(full_html)
print('Wrote standalone dashboard to', outpath)

## How to run the dashboard
- Option A (interactive Dash server):
  1. Ensure dependencies are installed (see the pip cell).
  2. Run the Dash cell:
     - In a code cell: `app = create_dash(); app.run_server(debug=True, port=8050)`
  3. Open `http://127.0.0.1:8050` in your browser.

- Option B (standalone HTML):
  - Open `public/projects/maternal_health_risk_predictor/maternal_dashboard.html` in your browser (created by the fallback cell).